# 2. K-Nearest Neighbors (KNN)
### Classification based on nearby data points

**K-Nearest Neighbors (KNN)** is a non-parametric, instance-based (lazy learning) algorithm. It classifies a new sample based on the majority class among its $k$ closest neighbors in feature space.

#### Key Concepts:
1. **Distance Metrics**:
   - Euclidean Distance: $d(p, q) = \sqrt{\sum_{i=1}^n (p_i - q_i)^2}$
   - Manhattan Distance: $d(p, q) = \sum_{i=1}^n |p_i - q_i|$
   - Minkowski Distance: $d(p, q) = \left(\sum_{i=1}^n |p_i - q_i|^p\right)^{1/p}$
2. **Choice of $k$**:
   - Small $k$: Low bias, high variance (sensitive to noise / overfitting).
   - Large $k$: High bias, smoother boundaries (underfitting risk).
3. **Feature Scaling**: Critical because distance calculations are dominated by features with larger magnitudes.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
# Step 2: Load and Prepare Wine Dataset
wine = load_wine()
X = wine.data
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Step 3: Train and Evaluate KNN Classifier
k = 5
knn = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='euclidean')
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

print(f"KNN (k={k}) Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("
Classification Report:
", classification_report(y_test, y_pred, target_names=wine.target_names))


In [ ]:
# Step 4: Finding the Optimal K Value
k_range = range(1, 25)
train_scores = []
test_scores = []

for cur_k in k_range:
    clf = KNeighborsClassifier(n_neighbors=cur_k)
    clf.fit(X_train_scaled, y_train)
    train_scores.append(accuracy_score(y_train, clf.predict(X_train_scaled)))
    test_scores.append(accuracy_score(y_test, clf.predict(X_test_scaled)))

plt.figure(figsize=(9, 5))
plt.plot(k_range, train_scores, marker='o', label='Train Accuracy')
plt.plot(k_range, test_scores, marker='s', label='Test Accuracy')
plt.xlabel("Value of K")
plt.ylabel("Accuracy")
plt.title("KNN: Effect of K on Training and Testing Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Step 5: Visualizing 2D Decision Boundaries
X_2d = X_train_scaled[:, :2]
y_2d = y_train

knn_2d = KNeighborsClassifier(n_neighbors=5)
knn_2d.fit(X_2d, y_2d)

x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                     np.arange(y_min, y_max, 0.05))

Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.viridis)
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, cmap=plt.cm.viridis, edgecolors='k')
plt.xlabel("Feature 1 (Standardized)")
plt.ylabel("Feature 2 (Standardized)")
plt.title("KNN Decision Boundaries (k=5)")
plt.legend(handles=scatter.legend_elements()[0], labels=list(wine.target_names))
plt.show()


### Summary
- **Strengths**: Simple to understand, no training phase required (lazy learning), naturally handles multiclass data.
- **Weaknesses**: Computationally expensive at inference time ($O(N \cdot D)$), suffers from curse of dimensionality, sensitive to irrelevant features and unscaled data.
